In [5]:
import json 
from tqdm import tqdm 

def overlaps(a_start: int, a_end: int, b_start: int, b_end: int) -> bool:
    return not (a_end <= b_start or b_end <= a_start)

def split_paragraphs_with_offsets(text):
    paragraphs = []
    parts = text.split("\n")

    cursor = 0
    for part in parts:
        part = part.strip()
        if not part:
            continue

        start = text.find(part, cursor)
        end = start + len(part)

        paragraphs.append({
            "text": part,
            "start": start,
            "end": end
        })

        cursor = end

    return paragraphs

In [6]:
synth_path = "../../../synthetic_sampling/samples/synthetic_samples_complete.json"
real_path = "../../../synthetic_sampling/samples/total_real_data.json"
synth_out_path = "../../data/lacks_synthesis/lacks_synthesis_ner_train.json"
real_out_path = "../../data/lacks_synthesis/lacks_synthesis_ner_eval.json"

with open(synth_path, "r", encoding="utf-8") as f:
    span_data_all = json.load(f)

with open(real_path, "r", encoding="utf-8") as f:
    real_data_all = json.load(f)

synth_lacks_synth_paragraphs = []
real_lacks_synth_paragraphs = []

for doc_idx, item in enumerate(tqdm(span_data_all["Lacks synthesis"], desc="Processing 'Lacks synthesis'")):
    doc_text = item["document"]
    spans = item.get("spans", [])

    paragraphs = split_paragraphs_with_offsets(doc_text)

    for p_idx, para in enumerate(paragraphs):
        p_start = para["start"]
        p_end = para["end"]
        para_text = para["text"]

        for sp in spans:
            sp_start = int(sp["start"])
            sp_end = int(sp["end"])

            if overlaps(sp_start, sp_end, p_start, p_end):

                # Clip span to paragraph
                clipped_start = max(sp_start, p_start)
                clipped_end = min(sp_end, p_end)

                # Convert to paragraph-local indices
                local_start = clipped_start - p_start
                local_end = clipped_end - p_start

                synth_lacks_synth_paragraphs.append({
                    "doc_index": doc_idx,
                    "paragraph": para_text,
                    "span": para_text[local_start:local_end],
                    "start": local_start,
                    "end": local_end,
                })



for doc_idx, item in enumerate(tqdm(real_data_all["Lacks synthesis"], desc="Processing 'Lacks synthesis'")):
    doc_text = item["document"]
    spans = item.get("spans", [])

    paragraphs = split_paragraphs_with_offsets(doc_text)

    for p_idx, para in enumerate(paragraphs):
        p_start = para["start"]
        p_end = para["end"]
        para_text = para["text"]

        for sp in spans:
            sp_start = int(sp["start"])
            sp_end = int(sp["end"])

            if overlaps(sp_start, sp_end, p_start, p_end):

                # Clip span to paragraph
                clipped_start = max(sp_start, p_start)
                clipped_end = min(sp_end, p_end)

                # Convert to paragraph-local indices
                local_start = clipped_start - p_start
                local_end = clipped_end - p_start

                real_lacks_synth_paragraphs.append({
                    "doc_index": doc_idx,
                    "paragraph": para_text,
                    "span": para_text[local_start:local_end],
                    "start": local_start,
                    "end": local_end,
                })

print("Total Lacks synthesis paragraph-span pairs:", len(synth_lacks_synth_paragraphs))
print("Total Lacks synthesis paragraph-span pairs:", len(real_lacks_synth_paragraphs))

Processing 'Lacks synthesis': 100%|██████████| 22/22 [00:00<00:00, 47030.93it/s]

Total Lacks synthesis paragraph-span pairs: 303
Total Lacks synthesis paragraph-span pairs: 33


In [7]:
import re
from typing import List, Dict, Any, Tuple


def _whitespace_tokenize_with_offsets(text: str) -> Tuple[List[str], List[Tuple[int, int]]]:
    """
    Whitespace tokenizer + char offsets into `text`.
    Tokens preserve punctuation attached to words (same as your earlier NER setup).
    """
    tokens = []
    offsets = []
    for m in re.finditer(r"\S+", text):
        tokens.append(m.group())
        offsets.append((m.start(), m.end()))
    return tokens, offsets


def _overlaps(a_start: int, a_end: int, b_start: int, b_end: int) -> bool:
    return not (a_end <= b_start or b_end <= a_start)


def lacks_synth_samples_to_ner(
    samples,
    label_name = "LACKS_SYNTHESIS",
    text_key= "paragraph",
    span_start_key = "start",
    span_end_key = "end",
    keep_metadata = True,
):
    out: List[Dict[str, Any]] = []
    b_tag = f"B-{label_name}"
    i_tag = f"I-{label_name}"

    for idx, ex in enumerate(samples):
        text = ex.get(text_key, "")
        if not isinstance(text, str) or not text.strip():
            continue

        try:
            span_start = int(ex[span_start_key])
            span_end = int(ex[span_end_key])
        except Exception:
            # if no valid offsets, skip
            continue

        # clamp
        span_start = max(0, min(len(text), span_start))
        span_end = max(0, min(len(text), span_end))
        if span_start >= span_end:
            continue

        tokens, offsets = _whitespace_tokenize_with_offsets(text)
        labels = ["O"] * len(tokens)

        covered = []
        for i, (ts, te) in enumerate(offsets):
            if _overlaps(ts, te, span_start, span_end):
                covered.append(i)

        # If nothing overlaps, you can skip or keep as all-O.
        # For coherence training you likely want to SKIP these.
        if not covered:
            continue

        labels[covered[0]] = b_tag
        for j in covered[1:]:
            labels[j] = i_tag

        item = {"tokens": tokens, "labels": labels}

        if keep_metadata:
            item.update(
                {
                    "doc_index": ex.get("doc_index", None),
                    "text": text,  # original paragraph
                    "span_start": span_start,
                    "span_end": span_end,
                    "span_text": text[span_start:span_end],
                    "source_span_field": ex.get("span", None),
                    "sample_index": idx,
                }
            )

        out.append(item)

    return out


In [8]:
train_ner = lacks_synth_samples_to_ner(synth_lacks_synth_paragraphs, label_name="LACKS_SYNTHESIS")
eval_ner = lacks_synth_samples_to_ner(real_lacks_synth_paragraphs, label_name="LACKS_SYNTHESIS")

with open(synth_out_path, "w", encoding="utf-8") as f:
    json.dump(train_ner, f, indent=2, ensure_ascii=False)

with open(real_out_path, "w", encoding="utf-8") as f:
    json.dump(eval_ner, f, indent=2, ensure_ascii=False)


Run the code below if experimenting with sentence indices overlap instead of BIO tagging 

In [1]:
import json
import re
from tqdm import tqdm
from typing import List, Dict, Any, Tuple


# ----------------------------
# Utils
# ----------------------------
def overlaps(a_start: int, a_end: int, b_start: int, b_end: int) -> bool:
    return not (a_end <= b_start or b_end <= a_start)


def split_paragraphs_with_offsets(text: str) -> List[Dict[str, Any]]:
    """
    Splits on blank lines and returns:
    [{"text": para_text, "start": start_char, "end": end_char}, ...]
    """
    paragraphs = []
    parts = text.split("\n\n")

    cursor = 0
    for part in parts:
        part = part.strip()
        if not part:
            continue

        start = text.find(part, cursor)
        end = start + len(part)

        paragraphs.append({"text": part, "start": start, "end": end})
        cursor = end

    return paragraphs


def _whitespace_tokenize_with_offsets(text: str) -> Tuple[List[str], List[Tuple[int, int]]]:
    """
    Whitespace tokenizer + char offsets into `text`.
    Tokens preserve punctuation attached to words.
    Returns:
      tokens:  ["Following", "the", "taxonomy", ...]
      offsets: [(0,9), (10,13), (14,22), ...]   char spans in `text`
    """
    tokens = []
    offsets = []
    for m in re.finditer(r"\S+", text):
        tokens.append(m.group())
        offsets.append((m.start(), m.end()))
    return tokens, offsets


def bio_labels_to_token_spans(labels: List[str]) -> List[Tuple[int, int, str]]:
    """
    Convert BIO labels at TOKEN level into contiguous token spans.

    Returns list of (start_token, end_token_exclusive, entity_type)
      - start_token inclusive
      - end_token exclusive

    Handles common BIO issues:
      - I-X starting a span (treated as B-X)
      - I-X after B-Y/I-Y where X != Y starts new span
    """
    spans: List[Tuple[int, int, str]] = []

    def ent_type(lbl: str) -> str:
        if "-" in lbl:
            return lbl.split("-", 1)[1]
        return ""

    i = 0
    n = len(labels)

    prev_tag = "O"
    prev_type = ""

    while i < n:
        lbl = (labels[i] or "O").strip()

        if lbl == "O":
            prev_tag, prev_type = "O", ""
            i += 1
            continue

        # Normalize invalid I-* cases into B-*
        if lbl.startswith("I-"):
            cur_type = ent_type(lbl)
            if prev_tag not in ("B", "I") or prev_type != cur_type:
                lbl = f"B-{cur_type}"

        if lbl.startswith("B-"):
            cur_type = ent_type(lbl)
            start = i
            i += 1
            prev_tag, prev_type = "B", cur_type

            while i < n:
                nxt = (labels[i] or "O").strip()
                if nxt.startswith("I-") and ent_type(nxt) == cur_type:
                    i += 1
                    prev_tag, prev_type = "I", cur_type
                    continue
                break

            end = i
            spans.append((start, end, cur_type))
            continue

        # Any other weird label -> skip
        prev_tag, prev_type = "O", ""
        i += 1

    return spans


def token_spans_to_char_spans(
    token_spans: List[Tuple[int, int, str]],
    token_offsets: List[Tuple[int, int]],
) -> List[Dict[str, Any]]:
    """
    Convert token spans (start_token, end_token, type) into char spans using token_offsets.

    Returns list of dicts:
      {"start_token":..., "end_token":..., "start":..., "end":..., "label":...}
    Where start/end are character offsets into the SAME text used to make token_offsets.
    """
    out: List[Dict[str, Any]] = []
    for st, en, lab in token_spans:
        st = max(0, min(st, len(token_offsets)))
        en = max(0, min(en, len(token_offsets)))
        if st >= en:
            continue
        char_start = token_offsets[st][0]
        char_end = token_offsets[en - 1][1]
        out.append(
            {
                "label": lab,
                "start_token": int(st),
                "end_token": int(en),
                "start": int(char_start),
                "end": int(char_end),
            }
        )
    return out


def ner_examples_to_span_samples(
    ner_examples: List[Dict[str, Any]],
    text_key: str = "paragraph",
    tokens_key: str = "tokens",
    labels_key: str = "labels",
    doc_index_key: str = "doc_index",
    keep_only_label: str | None = None,  # e.g., "COHERENCE"
) -> List[Dict[str, Any]]:
    """
    Takes NER examples of the form:
      {
        "doc_index": ...,
        "paragraph": "...",
        "tokens": [...],
        "labels": [... BIO ...]
      }

    Produces span samples with start/end character offsets in paragraph text:
      {
        "doc_index": ...,
        "paragraph": "...",
        "span_text": "...",
        "start": <char_start>,
        "end": <char_end>,
        "start_token": <token_start>,
        "end_token": <token_end>,
        "label": <entity_type>
      }

    IMPORTANT:
    - This assumes the paragraph text corresponds to whitespace-joined tokens (or at least
      that whitespace tokenization of paragraph reproduces the same token boundaries).
    - If your tokens were produced by the same whitespace tokenizer, this will match.
    """
    out: List[Dict[str, Any]] = []

    for idx, ex in enumerate(tqdm(ner_examples, desc="BIO->(start,end) spans")):
        text = ex.get(text_key, "") or ex.get("text", "")
        tokens = ex.get(tokens_key, None)
        labels = ex.get(labels_key, None)

        if not isinstance(text, str) or not text.strip():
            continue
        if not isinstance(tokens, list) or not isinstance(labels, list) or len(tokens) != len(labels):
            continue

        # Create offsets by re-tokenizing the paragraph text the same way as before
        # (whitespace tokens with punctuation attached).
        tok2, offsets = _whitespace_tokenize_with_offsets(text)

        # Sanity check: tokenization should match your stored tokens.
        # If it doesn't match, we fall back to using stored token indices mapped by position,
        # but char offsets may be unreliable in that case.
        if tok2 != [str(t) for t in tokens]:
            print(
                f"[WARN] Token mismatch in example {idx}. "
                f"Offsets may be unreliable. (len(text)={len(text)}, len(tokens)={len(tokens)}, len(tok2)={len(tok2)})"
            )
            # Best-effort: align by min length
            min_len = min(len(tokens), len(offsets))
            labels_aligned = [str(l) for l in labels[:min_len]]
            offsets_aligned = offsets[:min_len]
        else:
            labels_aligned = [str(l) for l in labels]
            offsets_aligned = offsets

        token_spans = bio_labels_to_token_spans(labels_aligned)
        char_spans = token_spans_to_char_spans(token_spans, offsets_aligned)

        for sp in char_spans:
            ent = sp["label"]
            if keep_only_label is not None and ent != keep_only_label:
                continue

            cs, ce = sp["start"], sp["end"]
            out.append(
                {
                    "doc_index": ex.get(doc_index_key, None),
                    "paragraph": text,
                    "span_text": text[cs:ce],
                    "start": cs,
                    "end": ce,
                    "start_token": sp["start_token"],
                    "end_token": sp["end_token"],
                    "label": ent,
                    "sample_index": idx,
                }
            )

    return out


In [2]:


# ----------------------------
# Example usage: convert existing BIO NER json to span start/end
# ----------------------------
# If you already have NER JSON files (tokens + labels + paragraph),
# point these to them. This will output a list of spans per paragraph.

INPUT_NER_JSON = "../../../data/lacks_synthesis/lacks_synthesis_ner_train.json"
EVAL_NER_JSON = "../../../data/lacks_synthesis/lacks_synthesis_ner_eval.json"
OUTPUT_SPANS_TRAIN = "../../../data/lacks_synthesis/lacks_synthesis_ner_train.json"
OUTPUT_SPANS_EVAL = "../../../data/lacks_synthesis/lacks_synthesis_ner_eval.json"

with open(INPUT_NER_JSON, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(EVAL_NER_JSON, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# Convert: BIO -> (start,end) char offsets (and also token start/end)
train_spans = ner_examples_to_span_samples(train_data, text_key="text", keep_only_label="LACKS_SYNTHESIS")
eval_spans = ner_examples_to_span_samples(eval_data, text_key="text", keep_only_label="LACKS_SYNTHESIS")
# NOTE: in your earlier writer, you used key "text" for the paragraph; if yours uses "paragraph" change text_key.

with open(OUTPUT_SPANS_TRAIN, "w", encoding="utf-8") as f:
    json.dump(train_spans, f, indent=2, ensure_ascii=False)

with open(OUTPUT_SPANS_EVAL, "w", encoding="utf-8") as f:
    json.dump(eval_spans, f, indent=2, ensure_ascii=False)

print("Wrote spans for train:", len(train_spans))
print("Wrote spans for eval:", len(eval_spans))


BIO->(start,end) spans: 100%|██████████| 33/33 [00:00<00:00, 12314.24it/s]

Wrote spans for train: 332
Wrote spans for eval: 33
